# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [5]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
print("Ollama instalado.")

Ollama instalado.


In [6]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests
servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")
!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.3


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [9]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)

MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    51 seconds ago            


### **CLIENTE DE PYTHON**

In [10]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**




### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [44]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
**Diferencia entre RAM y almacenamiento en una computadora**

La RAM (Random Access Memory) y el almacenamiento de una computadora son dos conceptos fundamentales que juegan un papel crucial en el rendimiento y la eficiencia de un sistema informático.

**RAM (Random Access Memory)**

La RAM es un tipo de memoria volátil que permite que la computadora acceda rápidamente a la información almacenada. La RAM se utiliza para almacenar los datos y programas que la computadora está utilizando en ese momento. Cuando la computadora está encendida, la RAM se llena con la información que está utilizando, lo que permite que la computadora acceda a esta información rápidamente.

**Almacenamiento**

El almacenamiento, por otro lado, es un tipo de memoria no volátil que almacena los datos y programas de forma permanente. El almacenamiento puede ser de varios tipos, como:

* Disco duro (HD)
* Disco rígido (SSD)
* Memoria flash

In [46]:
print(response.model_dump_json(indent=2)) # para inspeccionar la respuesta completa

{
  "model": "llama3.2:3b",
  "created_at": "2026-10-14T02:19:34.712128351Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 7488517679,
  "load_duration": 9154231,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 39684000,
  "eval_count": 608,
  "eval_duration": 7294842000,
  "message": {
    "role": "assistant",
    "content": "**Diferencia entre RAM y almacenamiento en una computadora**\n\nLa RAM (Random Access Memory) y el almacenamiento de una computadora son dos conceptos fundamentales que juegan un papel crucial en el rendimiento y la eficiencia de un sistema informático.\n\n**RAM (Random Access Memory)**\n\nLa RAM es un tipo de memoria volátil que permite que la computadora acceda rápidamente a la información almacenada. La RAM se utiliza para almacenar los datos y programas que la computadora está utilizando en ese momento. Cuando la computadora está encendida, la RAM se llena con la información que está utilizando, lo que permite que la computadora acceda a 

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [48]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count
print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)
# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste

Tokens del prompt: 43
Tokens de la respuesta: 608
Tokens totales: 651


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [49]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

inicio = time.time()
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 6.12 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [50]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada

inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)

Modelo ligero (llama3.2:3b): 18.31 s — 651 tokens
Modelo grande (llama3.1:8b): 18.75 s — 305 tokens (de los cuales 11.29 s fueron carga del modelo)

Respuesta del modelo grande:
 La RAM (Memoria de Acceso Aleatorio) y el almacenamiento son dos componentes clave de una computadora que cumplen funciones diferentes.

La RAM es una memoria que permite al procesador acceder a la información de manera rápida y eficiente, ya que es una memoria temporal que se utiliza para ejecutar programas y aplicaciones en el momento. La RAM es responsable de almacenar datos que se utilizan actualmente en la computadora, como los datos que se están procesando en un programa o la información que se está visualizando en la pantalla. La cantidad de RAM que tiene una computadora determina cuántos programas y aplicaciones se pueden ejecutar al mismo tiempo y cómo se utilizan los recursos de la computadora.

El almacenamiento, por otro lado, se refiere a la capacidad de una computadora para almacenar datos y prog

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Genera 3 preguntas:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

### **CONFIGURACIÓN DEL ENTORNO**

##### INSTALAR OLLAMA

In [53]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [54]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.3


In [56]:
# Descargar el modelo ligero de Llama e inicializar el cliente

MODELO_LIGERO = "llama3.2:3b"
!ollama pull {MODELO_LIGERO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    
llama3.1:8b    46e0c10c039e    4.9 GB    2 hours ago               


##### **INSTALAR CLIENTE DE PYTHON**




In [57]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


### **CONSULTA DE PREGUNTAS**

**2. Consulta la primera pregunta:**

* Construye una lista llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).
* Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

### **DEFINICIÓN DE PREGUNTAS**

In [58]:
# Definir la lista de preguntas
preguntas = [
    "¿Cómo puedo restablecer mi contraseña si no tengo acceso al correo electrónico registrado?",
    "¿Cuáles son los métodos de pago aceptados en la tienda online?",
    "¿Cuál es el horario de atención al cliente y cómo puedo contactar con soporte técnico?"
]

print(f"Lista de preguntas creada con {len(preguntas)} preguntas.")
for i, p in enumerate(preguntas, 1):
    print(f"  {i}. {p}")

Lista de preguntas creada con 3 preguntas.
  1. ¿Cómo puedo restablecer mi contraseña si no tengo acceso al correo electrónico registrado?
  2. ¿Cuáles son los métodos de pago aceptados en la tienda online?
  3. ¿Cuál es el horario de atención al cliente y cómo puedo contactar con soporte técnico?


#### **PREGUNTA 1**

In [63]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1
import time

inicio = time.time()

respuesta_1 = client.chat(
    model=MODELO_LIGERO,
    messages=[{'role': 'user', 'content': preguntas[0]}]
)

fin = time.time()
tiempo_1 = fin - inicio

tokens_prompt_1 = respuesta_1.get('prompt_eval_count', 0)
tokens_respuesta_1 = respuesta_1.get('eval_count', 0)
tokens_totales_1 = tokens_prompt_1 + tokens_respuesta_1

resultado_1 = {
    'pregunta': preguntas[0],
    'respuesta': respuesta_1['message']['content'],
    'tiempo_segundos': round(tiempo_1, 2),
    'tokens_prompt': tokens_prompt_1,
    'tokens_respuesta': tokens_respuesta_1,
    'tokens_totales': tokens_totales_1
}

print(f"Pregunta 1: {resultado_1['pregunta']}")
print(f"⏱️  Tiempo: {resultado_1['tiempo_segundos']}s")
print(f"🔢 Tokens totales: {resultado_1['tokens_totales']}")
print(f"\n📝 Respuesta:\n{resultado_1['respuesta']}")


Pregunta 1: ¿Cómo puedo restablecer mi contraseña si no tengo acceso al correo electrónico registrado?
⏱️  Tiempo: 4.93s
🔢 Tokens totales: 450

📝 Respuesta:
Si no tienes acceso al correo electrónico registrado que estás asociado con tu cuenta, es posible que debas utilizar otros métodos para restablecer tu contraseña. Aquí te presento algunas opciones:

1. **Contacta con el soporte técnico de la plataforma o servicio**: La mayoría de las plataformas y servicios de línea te ofrecen un proceso de restablecimiento de contraseña a través de su soporte técnico. Puedes buscar el número de teléfono o la dirección de correo electrónico de soporte técnico de la plataforma y contactarlos para obtener ayuda.
2. **Utiliza la función de recuperación de contraseña**: Muchas plataformas y servicios de línea ofrecen una función de recuperación de contraseña que te permite restablecer tu contraseña sin necesidad de tu correo electrónico. Busca la sección de "Recuperar contraseña" o "Olvidé mi contraseñ

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

#### **PREGUNTA 2**

In [60]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2
import time

inicio = time.time()

respuesta_2 = client.chat(
    model=MODELO_LIGERO,
    messages=[{'role': 'user', 'content': preguntas[1]}]
)

fin = time.time()
tiempo_2 = fin - inicio

tokens_prompt_2 = respuesta_2.get('prompt_eval_count', 0)
tokens_respuesta_2 = respuesta_2.get('eval_count', 0)
tokens_totales_2 = tokens_prompt_2 + tokens_respuesta_2

resultado_2 = {
    'pregunta': preguntas[1],
    'respuesta': respuesta_2['message']['content'],
    'tiempo_segundos': round(tiempo_2, 2),
    'tokens_prompt': tokens_prompt_2,
    'tokens_respuesta': tokens_respuesta_2,
    'tokens_totales': tokens_totales_2
}

print(f"Pregunta 2: {resultado_2['pregunta']}")
print(f"⏱️  Tiempo: {resultado_2['tiempo_segundos']}s")
print(f"🔢 Tokens totales: {resultado_2['tokens_totales']}")
print(f"\n📝 Respuesta:\n{resultado_2['respuesta']}")

Pregunta 2: ¿Cuáles son los métodos de pago aceptados en la tienda online?
⏱️  Tiempo: 5.48s
🔢 Tokens totales: 495

📝 Respuesta:
Excelente pregunta!

En una tienda online, generalmente se aceptan varios métodos de pago para facilitar las transacciones a los clientes. Algunos de los métodos de pago más comunes son:

1. **Tarjetas de crédito**: Visa, Mastercard, American Express, Discover, entre otras.
2. **Tarjetas de débito**: Tarjetas de débito Visa, Mastercard, American Express, entre otras.
3. **PayPal**: Es uno de los métodos de pago más populares en línea, que permite a los clientes pagar sin necesidad de proporcionar información de tarjeta.
4. **PayPal Express**: Es una forma de pago en línea que utiliza PayPal como intermediario.
5. **Transferencia bancaria**: La mayoría de las tiendas online aceptan transferencias bancarias, que permiten a los clientes transferir fondos de su cuenta bancaria a la cuenta de la tienda.
6. **Cheques**: Algunas tiendas online aceptan cheques como m

#### **PREGUNTA 3**

In [61]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3
import time

inicio = time.time()

respuesta_3 = client.chat(
    model=MODELO_LIGERO,
    messages=[{'role': 'user', 'content': preguntas[2]}]
)

fin = time.time()
tiempo_3 = fin - inicio

tokens_prompt_3 = respuesta_3.get('prompt_eval_count', 0)
tokens_respuesta_3 = respuesta_3.get('eval_count', 0)
tokens_totales_3 = tokens_prompt_3 + tokens_respuesta_3

resultado_3 = {
    'pregunta': preguntas[2],
    'respuesta': respuesta_3['message']['content'],
    'tiempo_segundos': round(tiempo_3, 2),
    'tokens_prompt': tokens_prompt_3,
    'tokens_respuesta': tokens_respuesta_3,
    'tokens_totales': tokens_totales_3
}

print(f"Pregunta 3: {resultado_3['pregunta']}")
print(f"⏱️  Tiempo: {resultado_3['tiempo_segundos']}s")
print(f"🔢 Tokens totales: {resultado_3['tokens_totales']}")
print(f"\n📝 Respuesta:\n{resultado_3['respuesta']}")


Pregunta 3: ¿Cuál es el horario de atención al cliente y cómo puedo contactar con soporte técnico?
⏱️  Tiempo: 4.35s
🔢 Tokens totales: 405

📝 Respuesta:
Lo siento, pero no puedo proporcionar información específica sobre horarios de atención al cliente o contactos de soporte técnico para empresas o servicios específicos. Sin embargo, puedo ofrecerte algunas sugerencias generales sobre cómo encontrar esta información:

1. **Visita el sitio web de la empresa**: La mayoría de las empresas tienen un sitio web donde proporcionan información sobre su soporte técnico, horarios de atención al cliente y cómo contactar con ellos. Busca secciones como "Soporte técnico", "Ayuda", "Contacto" o "Sobre nosotros" en el menú superior.

2. **Revisa las redes sociales**: Las empresas a menudo publican información sobre su soporte técnico y horarios de atención en sus perfiles en redes sociales como Twitter, Facebook o Instagram.

3. **Busca en Google**: Si no puedes encontrar la información en el sitio we

#### **CONJUNCIÓN Y SÍNTESIS DE RESULTADOS**

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [62]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

print(f"Lista 'resultados' creada con {len(resultados)} elementos.")
for i, r in enumerate(resultados, 1):
    print(f"  {i}. Pregunta: {r['pregunta'][:60]}... | ⏱️ {r['tiempo_segundos']}s | 🔢 {r['tokens_totales']} tokens")

Lista 'resultados' creada con 3 elementos.
  1. Pregunta: ¿Cómo puedo restablecer mi contraseña si no tengo acceso al ... | ⏱️ 71.23s | 🔢 473 tokens
  2. Pregunta: ¿Cuáles son los métodos de pago aceptados en la tienda onlin... | ⏱️ 5.48s | 🔢 495 tokens
  3. Pregunta: ¿Cuál es el horario de atención al cliente y cómo puedo cont... | ⏱️ 4.35s | 🔢 405 tokens


**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [64]:
# Mostrar la tabla final de resultados
print("=" * 90)
print("📊 TABLA COMPARATIVA - MODELO LIGERO (llama3.2:3b)")
print("=" * 90)
print(f"{'#':<3} {'Pregunta':<50} {'Tiempo (s)':<12} {'Tokens':<10}")
print("-" * 90)

for i, r in enumerate(resultados, 1):
    pregunta_corta = r['pregunta'][:48] + "..." if len(r['pregunta']) > 50 else r['pregunta']
    print(f"{i:<3} {pregunta_corta:<50} {r['tiempo_segundos']:<12} {r['tokens_totales']:<10}")

print("-" * 90)

# Calcular totales y promedios
tiempo_total = sum(r['tiempo_segundos'] for r in resultados)
tokens_total = sum(r['tokens_totales'] for r in resultados)
tiempo_promedio = tiempo_total / len(resultados)

print(f"{'':<3} {'TOTAL':<50} {tiempo_total:.2f} {'':<12} {tokens_total:<10}")
print(f"{'':<3} {'PROMEDIO':<50} {tiempo_promedio:.2f}")
print("=" * 90)

# Conclusión
print("\n📝 CONCLUSIÓN:")
todas_respondidas = all(len(r['respuesta'].strip()) > 0 for r in resultados)

if todas_respondidas:
    print("✅ El modelo ligero (llama3.2:3b) resolvió las 3 preguntas SATISFACTORIAMENTE.")
    print(f"   - Todas las preguntas recibieron una respuesta completa.")
    print(f"   - Tiempo promedio por pregunta: {tiempo_promedio:.2f} segundos.")
    print(f"   - Tokens totales consumidos: {tokens_total}.")
else:
    print("⚠️ El modelo ligero no resolvió todas las preguntas correctamente.")

📊 TABLA COMPARATIVA - MODELO LIGERO (llama3.2:3b)
#   Pregunta                                           Tiempo (s)   Tokens    
------------------------------------------------------------------------------------------
1   ¿Cómo puedo restablecer mi contraseña si no teng... 71.23        473       
2   ¿Cuáles son los métodos de pago aceptados en la ... 5.48         495       
3   ¿Cuál es el horario de atención al cliente y cóm... 4.35         405       
------------------------------------------------------------------------------------------
    TOTAL                                              81.06              1373      
    PROMEDIO                                           27.02

📝 CONCLUSIÓN:
✅ El modelo ligero (llama3.2:3b) resolvió las 3 preguntas SATISFACTORIAMENTE.
   - Todas las preguntas recibieron una respuesta completa.
   - Tiempo promedio por pregunta: 27.02 segundos.
   - Tokens totales consumidos: 1373.
